In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/sehakflower/data/main/titanic.csv"

df = pd.read_csv(url, sep='\t')
df_clean = df.copy()

# print('[결측지]', "\n", df.isnull().sum())
print(df.head(3))

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  


In [2]:
df['Title'] = df['Name'].str.split(',').str[1].str.split('.').str[0].str.strip()

print(df[['Name', 'Title']].head(5))

                                                Name Title
0                            Braund, Mr. Owen Harris    Mr
1  Cumings, Mrs. John Bradley (Florence Briggs Th...   Mrs
2                             Heikkinen, Miss. Laina  Miss
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)   Mrs
4                           Allen, Mr. William Henry    Mr


In [3]:
def clean_title(title):
    if title == 'Mr':
        return 0
    elif title in ['Miss', 'Mlle', 'Ms']:
        return 1
    elif title in ['Mrs', 'Mme']:
        return 2
    elif title == 'Master':
        return 3
    else:
        return 4

print("Mr를 넣었을 때의 결과:", clean_title('Mr'))
print("Dr(의사)를 넣었을 때의 결과:", clean_title('Dr'))

Mr를 넣었을 때의 결과: 0
Dr(의사)를 넣었을 때의 결과: 4


In [4]:
df['Title_num'] = df['Title'].apply(clean_title)

print(df[['Name', 'Title', 'Title_num']].head(5))

                                                Name Title  Title_num
0                            Braund, Mr. Owen Harris    Mr          0
1  Cumings, Mrs. John Bradley (Florence Briggs Th...   Mrs          2
2                             Heikkinen, Miss. Laina  Miss          1
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)   Mrs          2
4                           Allen, Mr. William Henry    Mr          0


In [5]:
호칭별_나이_중앙값 = df.groupby('Title')['Age'].median()

print(호칭별_나이_중앙값)

Title
Don       40.0
Master     4.0
Miss      17.5
Mr        28.0
Mrs       31.0
Rev       46.5
Name: Age, dtype: float64


In [6]:
df['Age'] = df['Age'].fillna(df.groupby('Title')['Age'].transform('median'))

print("Age 열의 남은 빈칸 개수:", df['Age'].isnull().sum())

Age 열의 남은 빈칸 개수: 0


In [7]:
age_bins = [0, 16, 32, 48, 64, 100]

age_labels = [0, 1, 2, 3, 4]

df['Age_group'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels)

print(df[['Age', 'Age_group']].head(5))

    Age Age_group
0  22.0         1
1  38.0         2
2  26.0         1
3  35.0         2
4  35.0         2


In [8]:
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

print("Fare 열의 남은 빈칸 개수:", df['Fare'].isnull().sum())

Fare 열의 남은 빈칸 개수: 0


In [9]:
fare_labels = [0, 1, 2, 3]

df['Fare_group'] = pd.qcut(df['Fare'], q=4, labels=fare_labels)

print(df['Fare_group'].value_counts())

Fare_group
1    40
0    39
3    39
2    38
Name: count, dtype: int64


In [10]:
df['Cabin'] = df['Cabin'].fillna('N')
df['Cabin_sector'] = df['Cabin'].str[0]
print(df['Cabin_sector'].value_counts())

Cabin_sector
N    125
C     10
D      6
B      5
F      4
E      3
A      2
G      1
Name: count, dtype: int64


In [11]:
cabin_mapping = {
    'A': 0,
    'B': 1,
    'C': 2,
    'D': 3,
    'E': 4,
    'F': 5,
    'G': 6,
    'N': 7
}
df['Cabin_num'] = df['Cabin_sector'].map(cabin_mapping)

df['Cabin_num'] = df['Cabin_num'].fillna(7)
print(df[['Cabin_sector', 'Cabin_num']].head(8))

  Cabin_sector  Cabin_num
0            N          7
1            C          2
2            N          7
3            C          2
4            N          7
5            N          7
6            E          4
7            N          7


In [12]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['Embarked'] = df['Embarked'].fillna('S')
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['fare_person'] = df['Fare'] / df['FamilySize']

print(df[['Sex', 'Embarked', 'FamilySize', 'fare_person']].head())

   Sex  Embarked  FamilySize  fare_person
0    0         0           2      3.62500
1    1         1           2     35.64165
2    1         0           1      7.92500
3    1         0           2     26.55000
4    0         0           1      8.05000
